# EchoNext Publication Pipeline

This notebook is the publication-oriented Colab workflow. It uses:
- GitHub as the source of truth for the code
- Google Drive as the source of truth for the dataset and saved outputs
- Colab local runtime storage for training speed

You can run this notebook top to bottom with `Runtime -> Run all`.

## 1. Runtime setup

Recommended Colab settings:
- Runtime type: Python 3
- Hardware accelerator: L4 GPU or T4 GPU
- High-RAM: on

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/echonext_single_lead')
LOCAL_DATA_DIR = PROJECT_DIR / 'data' / 'raw'
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/echonext-a-dataset-for-detecting-echocardiogram-confirmed-structural-heart-disease-from-ecgs-1.1.1')
DRIVE_RESULTS_DIR = Path('/content/drive/MyDrive/echonext_outputs_publication')

LABEL = 'shd_moderate_or_greater_flag'
SEEDS = '42 43 44'
EPOCHS = 5
BATCH_SIZE = 16
BOOTSTRAP_ITERATIONS = 1000

print('PROJECT_DIR =', PROJECT_DIR)
print('LOCAL_DATA_DIR =', LOCAL_DATA_DIR)
print('DRIVE_DATA_DIR =', DRIVE_DATA_DIR)
print('DRIVE_RESULTS_DIR =', DRIVE_RESULTS_DIR)
print('LABEL =', LABEL)
print('SEEDS =', SEEDS)
print('EPOCHS =', EPOCHS)
print('BATCH_SIZE =', BATCH_SIZE)
print('BOOTSTRAP_ITERATIONS =', BOOTSTRAP_ITERATIONS)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/echonext_single_lead
!git clone https://github.com/aavascular/echonext_single_lead.git /content/echonext_single_lead
!ls /content/echonext_single_lead


In [ ]:
!pip install -r /content/echonext_single_lead/requirements.txt


In [ ]:
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 2. Verify and copy required data

In [ ]:
required_files = [
    'echonext_metadata_100k.csv',
    'EchoNext_train_waveforms.npy',
    'EchoNext_val_waveforms.npy',
    'EchoNext_test_waveforms.npy',
    'EchoNext_train_tabular_features.npy',
    'EchoNext_val_tabular_features.npy',
    'EchoNext_test_tabular_features.npy',
]

print('Drive data directory exists:', DRIVE_DATA_DIR.exists())
missing = [name for name in required_files if not (DRIVE_DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')
print('All required Drive files found.')


In [ ]:
import shutil

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
for name in required_files:
    src = DRIVE_DATA_DIR / name
    dst = LOCAL_DATA_DIR / name
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print('Skipping existing file:', name)
        continue
    print('Copying', name)
    shutil.copy2(src, dst)

print('\nLocal data directory contents:')
for path in sorted(LOCAL_DATA_DIR.glob('*')):
    print(' -', path.name)


## 3. Inspect the dataset

In [ ]:
%cd /content/echonext_single_lead
!python3 scripts/01_inspect_data.py --data_dir data/raw


## 4. Run the single-lead benchmark sweep

This runs:
- tabular only
- full 12-lead
- six limb leads
- Lead I + tabular
- each single lead separately

In [ ]:
!python3 scripts/04_run_lead_sweep.py --data_dir data/raw --label {LABEL} --epochs {EPOCHS} --batch_size {BATCH_SIZE}


## 5. Run repeated-seed stability for the core publication models

This runs 3 seeds each for:
- tabular only
- Lead I
- Lead I + tabular
- six limb leads
- full 12-lead

In [ ]:
!python3 scripts/07_run_seed_stability.py --data_dir data/raw --label {LABEL} --seeds {SEEDS} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --include_six_limb


## 6. Build publication-style core summaries and confidence intervals

In [ ]:
!python3 scripts/08_make_publication_results.py --label {LABEL} --seeds {SEEDS} --include_six_limb --bootstrap_iterations {BOOTSTRAP_ITERATIONS}


## 7. Build aggregate benchmark tables and figures

In [ ]:
!python3 scripts/06_make_results_tables.py --output_dir outputs


## 8. Inspect the generated output files

In [ ]:
!find outputs -maxdepth 3 -type f | sort


In [ ]:
import pandas as pd

publication_core_df = pd.read_csv('outputs/tables/publication_core_results.csv')
seed_stability_df = pd.read_csv('outputs/tables/seed_stability_summary.csv')
benchmark_df = pd.read_csv('outputs/tables/model_performance_by_input.csv')

print('Publication core results')
display(publication_core_df)

print('Seed stability summary')
display(seed_stability_df)

print('Benchmark table')
display(benchmark_df)


In [ ]:
from IPython.display import Image, display

for figure_path in [
    'outputs/figures/auroc_by_input.png',
    'outputs/figures/auprc_by_input.png',
    'outputs/figures/calibration_plot.png',
    'outputs/figures/seed_stability_auroc.png',
    'outputs/figures/publication_core_roc.png',
    'outputs/figures/publication_core_pr.png',
    'outputs/figures/publication_core_auroc_ci.png',
    'outputs/figures/publication_core_auprc_ci.png',
]:
    print('\n', figure_path)
    display(Image(filename=figure_path))


## 9. Copy all outputs back to Google Drive

In [ ]:
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
!rm -rf /content/drive/MyDrive/echonext_outputs_publication/outputs
!cp -r /content/echonext_single_lead/outputs /content/drive/MyDrive/echonext_outputs_publication/
!find /content/drive/MyDrive/echonext_outputs_publication/outputs -maxdepth 3 -type f | sort
